# A.2

In [1]:
import pandas as pd

data_path = 'data/competitive-data-science-predict-future-sales/'

sales_train = pd.read_csv(data_path + 'sales_train.csv')
shops = pd.read_csv(data_path + 'shops.csv')
items = pd.read_csv(data_path + 'items.csv')
item_categories = pd.read_csv(data_path + 'item_categories.csv')

In [2]:
train = sales_train.merge(shops, on='shop_id', how='left')
train = train.merge(items, on='item_id', how='left')
train = train.merge(item_categories, on='item_category_id', how='left')

In [3]:
train.dtypes

date                   object
date_block_num          int64
shop_id                 int64
item_id                 int64
item_price            float64
item_cnt_day          float64
shop_name              object
item_name              object
item_category_id        int64
item_category_name     object
dtype: object

In [4]:
train.memory_usage()

Index                      132
date                  23486792
date_block_num        23486792
shop_id               23486792
item_id               23486792
item_price            23486792
item_cnt_day          23486792
shop_name             23486792
item_name             23486792
item_category_id      23486792
item_category_name    23486792
dtype: int64

In [5]:
start_mem = train.memory_usage().sum() / 1024**2
start_mem

np.float64(223.98762893676758)

In [6]:
for col in train.columns:
    dtype_name = train[col].dtype.name
    if dtype_name == 'object':
        pass
    elif dtype_name == 'bool':
        train[col] = train[col].astype('int8')
    elif dtype_name.startswith('int') or (train[col].round()==train[col]).all():
        train[col] = pd.to_numeric(train[col], downcast='integer')
    else:
        train[col] = pd.to_numeric(train[col], downcast='float')

In [7]:
train.dtypes

date                   object
date_block_num           int8
shop_id                  int8
item_id                 int16
item_price            float64
item_cnt_day            int16
shop_name              object
item_name              object
item_category_id         int8
item_category_name     object
dtype: object

In [8]:
train.memory_usage()

Index                      132
date                  23486792
date_block_num         2935849
shop_id                2935849
item_id                5871698
item_price            23486792
item_cnt_day           5871698
shop_name             23486792
item_name             23486792
item_category_id       2935849
item_category_name    23486792
dtype: int64

In [9]:
end_mem = train.memory_usage().sum() / 1024**2
end_mem

np.float64(131.59278392791748)

In [10]:
print('{:.1f}% 압축됨'.format(100 * (start_mem - end_mem) / start_mem))

41.2% 압축됨


In [11]:
def downcast(df, verbose=True):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        dtype_name = df[col].dtype.name
        if dtype_name == 'object':
            pass
        elif dtype_name == 'bool':
            df[col] = df[col].astype('int8')
        elif dtype_name.startswith('int') or (df[col].round() == df[col]).all():
            df[col] = pd.to_numeric(df[col], downcast='integer')
        else:
            df[col] = pd.to_numeric(df[col], downcast='float')
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print('{:.1f}% 압축됨'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

# A.3

In [12]:
import pandas as pd

data_path = 'data/plant-pathology-2020-fgvc7/'

DEBUG = True

if DEBUG:
    nrows=200
    epochs=1
else:
    nrows=None
    epochs=39

train = pd.read_csv(data_path + 'train.csv', nrows=nrows)
test = pd.read_csv(data_path + 'test.csv')
submission = pd.read_csv(data_path + 'sample_submission.csv')

# A.4

In [16]:
import torch

path = 'data/plant-pathology-2020-fgvc7/'

torch.save({
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict()
}, path + 'EfficientNet-B7.tar')

In [13]:
!pip install efficientnet-pytorch

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16522 sha256=157deafb1b40a1023addeb0404c8ae481b71ca153081bf04dda2fe8ced8a27b5
  Stored in directory: c:\users\지우\appdata\local\pip\cache\wheels\5b\40\c3\afeb111ff7a5abde37eca97b3b447651c6de127933eef6941f
Successfully built efficientnet-pytorch



[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from efficientnet_pytorch import EfficientNet

model = EfficientNet.from_pretrained('efficientnet-b7', num_classes=4)

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b7-dcc49843.pth" to C:\Users\지우/.cache\torch\hub\checkpoints\efficientnet-b7-dcc49843.pth
100%|██████████| 254M/254M [01:27<00:00, 3.06MB/s] 


Loaded pretrained weights for efficientnet-b7


In [15]:
import torch

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00006, weight_decay=0.0001)

In [17]:
pretrained_model_path = 'data/plant-pathology-2020-fgvc7/'

checkpoint = torch.load(pretrained_model_path + 'EfficientNet-B7.tar')

model.load_state_dict(checkpoint['model'])
optimizer.load_state_dict(checkpoint['optimizer'])